# Resume Analyzer AI
Paste your full code here.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pdfminer.high_level import extract_text
import io, re, nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

stop_words = set(stopwords.words('english'))

SKILL_BANK = [
    'python','machine learning','sql','tensorflow','data analysis','pandas',
    'numpy','excel','power bi','statistics','react','javascript','html','css',
    'node.js','mongodb','git','keras','deep learning','nlp','aws','docker',
    'flask','django','java','c++','tableau','scikit-learn','pytorch','spark',
    'communication','leadership','project management','agile','scrum',
    'figma','photoshop','ui','ux','kubernetes','linux','rest api','typescript',
    'arduino','raspberry pi','mqtt','iot','embedded systems','c','rtos',
    'sensors','microcontroller','zigbee','bluetooth','wifi','esp32','plc',
    'research','literature review','matlab','latex','data collection',
    'academic writing','hypothesis','experimentation','r','statistics',
    'signal processing','circuit design','pcb','fpga','verilog','simulink'
]

JOB_DESCRIPTIONS = {
    "🤖  Data Scientist": """
        We are looking for a Data Scientist with strong experience in Python, Machine Learning,
        TensorFlow, and Deep Learning. The candidate must have expertise in data analysis,
        pandas, numpy, and scikit-learn. Experience with NLP and PyTorch is a plus.
        SQL knowledge is required. Must have 2+ years of experience building ML models.
    """,
    "🌐  Full Stack Web Developer": """
        Seeking a Full Stack Developer with experience in React, JavaScript, TypeScript,
        HTML, and CSS. Backend skills in Node.js, Django or Flask required. Database
        experience with MongoDB and SQL needed. Familiarity with REST API, Git, Docker, AWS.
    """,
    "📊  Business Data Analyst": """
        Looking for a Data Analyst proficient in SQL, Excel, Power BI, and Tableau.
        Strong skills in data analysis, statistics, and Python (pandas, numpy).
        Experience creating dashboards required. Strong communication skills needed.
    """,
    "☁️  DevOps / Cloud Engineer": """
        Need a DevOps Engineer with experience in AWS, Docker, Kubernetes, and Linux.
        Strong scripting in Python required. Familiar with CI/CD pipelines, Git,
        REST APIs, and agile practices. Cloud deployment experience essential.
    """,
    "🎨  UI/UX Designer": """
        Looking for a UI/UX Designer with expertise in Figma, Photoshop, UI and UX.
        Experience with wireframes and prototyping essential. HTML and CSS knowledge a plus.
        Excellent communication skills needed. Portfolio of past design work required.
    """,
    "⚙️  ML Engineer": """
        Seeking an ML Engineer with expertise in Python, TensorFlow, PyTorch, Keras,
        and scikit-learn. Must deploy ML models using Flask or Docker on AWS.
        Strong understanding of deep learning, NLP, pandas, numpy. SQL and Git required.
    """,
    "📱  Backend Developer": """
        Hiring a Backend Developer with strong Python skills in Django and Flask.
        Knowledge of REST API, SQL, MongoDB, and Git required. Docker, AWS, Linux a plus.
        Must know agile and scrum. 2+ years backend development experience required.
    """,
    "🔌  IoT Engineer": """
        Looking for an IoT Engineer with experience in embedded systems, Arduino,
        Raspberry Pi, and ESP32. Knowledge of MQTT, Bluetooth, WiFi, Zigbee required.
        Proficiency in C and Python essential. Experience with sensors, RTOS, PCB,
        and circuit design a strong plus. Cloud IoT integration preferred.
    """,
    "🔬  Research Intern": """
        Looking for a Research Intern with experience in literature review, data collection,
        experimentation, and academic writing. Proficiency in Python, MATLAB, R required.
        Knowledge of LaTeX a plus. Hypothesis formulation and research methodology preferred.
        Strong communication skills essential. Currently pursuing a relevant degree.
    """
}

# ── ML FUNCTIONS ─────────────────────────────────────────────
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [w for w in text.split() if w not in stop_words]
    return ' '.join(words)

def extract_skills(text):
    text = text.lower()
    return [s for s in SKILL_BANK if s in text]

def extract_years(text):
    matches = re.findall(r'(\d+)\s*\+?\s*years?', text.lower())
    return max([int(m) for m in matches], default=0)

def get_fit_verdict(score, matched_count, jd_skill_count):
    if score >= 40:
        return "🌟 Ideal Fit", "#00e5ff", "#071318", "#00e5ff33"
    elif score >= 20:
        return "⚡ Average Fit", "#a78bfa", "#0d0b18", "#7c3aed33"
    elif score >= 10:
        return "🙂 You Can Try", "#fbbf24", "#18130a", "#fbbf2433"
    else:
        return "❌ Not a Good Fit", "#ef4444", "#180b0b", "#ef444433"

def analyze_resumes(resumes, job_description):
    cleaned_jd = clean_text(job_description)
    jd_skills = extract_skills(job_description)
    results = []
    all_texts = [cleaned_jd] + [clean_text(t) for t in resumes.values()]
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(all_texts)
    scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    for (name, text), score in zip(resumes.items(), scores):
        skills = extract_skills(text)
        matched = [s for s in skills if s in jd_skills]
        missing = [s for s in jd_skills if s not in skills]
        years = extract_years(text)
        fit_label, fit_color, fit_bg, fit_border = get_fit_verdict(score * 100, len(matched), len(jd_skills) or 1)
        results.append({
            'name': name, 'score': round(score * 100, 1),
            'skills': skills, 'matched': matched, 'missing': missing,
            'years': years, 'fit_label': fit_label, 'fit_color': fit_color,
            'fit_bg': fit_bg, 'fit_border': fit_border, 'jd_skill_count': len(jd_skills)
        })
    return sorted(results, key=lambda x: x['score'], reverse=True)

# ── STYLES ───────────────────────────────────────────────────
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=DM+Mono:wght@400;500&display=swap');
.ra-title { font-family: Syne,sans-serif; font-size: 2rem; font-weight: 800; color: #00e5ff; margin-bottom: 4px; }
.ra-sub { font-family: DM Mono,monospace; color: #555; font-size: 0.8rem; margin-bottom: 20px; }
.ra-label { font-family: Syne,sans-serif; font-size: 0.72rem; text-transform: uppercase; letter-spacing: 2px; color: #7c3aed; font-weight: 700; margin: 20px 0 8px 0; display: block; }
.ra-card { background: #0f0f1a; border: 1px solid #1e1e35; border-radius: 12px; padding: 20px; margin-bottom: 14px; }
.ra-card-top { display: flex; align-items: flex-start; gap: 16px; }
.ra-rank { font-size: 1.5rem; min-width: 36px; }
.ra-info { flex: 1; }
.ra-cname { font-family: Syne,sans-serif; font-weight: 700; font-size: 1.05rem; margin-bottom: 6px; color: #f0f0f0; }
.skill-tag { display: inline-block; background: #1a1a2e; border: 1px solid #2a2a4a; border-radius: 4px; padding: 2px 8px; margin: 2px; font-size: 0.68rem; color: #00e5ff; font-family: DM Mono,monospace; }
.skill-tag.matched { background: #1a0a2e; border-color: #7c3aed; color: #a78bfa; }
.skill-tag.missing { background: #1a0808; border-color: #ef444466; color: #ef4444; }
.ra-score-bar { width: 110px; flex-shrink: 0; text-align: right; }
.ra-score-num { font-size: 1.5rem; font-weight: 800; font-family: DM Mono,monospace; color: #00e5ff; }
.bar-bg { background: #1a1a2e; border-radius: 99px; height: 5px; margin-top: 6px; overflow: hidden; }
.bar-fill { height: 100%; border-radius: 99px; background: #7c3aed; }
.ra-badge { display: inline-block; background: #7c3aed22; border: 1px solid #7c3aed44; border-radius: 4px; padding: 2px 8px; font-size: 0.66rem; color: #a78bfa; font-family: DM Mono,monospace; margin-left: 6px; }
.fit-box { border-radius: 8px; padding: 10px 14px; margin-top: 12px; font-family: DM Mono,monospace; font-size: 0.74rem; line-height: 1.9; }
.fit-verdict { font-family: Syne,sans-serif; font-size: 0.95rem; font-weight: 700; margin-bottom: 4px; }
.winner-banner { background: #0d0d1a; border: 1px solid #00e5ff33; border-left: 3px solid #00e5ff; border-radius: 10px; padding: 14px 20px; margin-bottom: 16px; font-family: Syne,sans-serif; font-size: 0.88rem; color: #e0e0e0; }
.ra-divider { border: none; border-top: 1px solid #1a1a2e; margin: 16px 0; }
.ra-empty { text-align: center; padding: 30px; color: #444; font-family: DM Mono,monospace; font-size: 0.82rem; background: #0f0f1a; border-radius: 12px; border: 1px solid #1e1e35; }
.step-note { font-family: DM Mono,monospace; font-size: 0.73rem; color: #444; background: #0d0d1a; border: 1px solid #1a1a2e; border-radius: 8px; padding: 10px 14px; margin-top: 6px; line-height: 1.8; }
</style>
"""))

# ── STATE ────────────────────────────────────────────────────
resume_store = {}
output_area  = widgets.Output()
load_status  = widgets.Output()
jd_status    = widgets.Output()

# ── HEADER ───────────────────────────────────────────────────
display(HTML("<div class='ra-title'>Resume Analyzer</div>"))
display(HTML("<div class='ra-sub'>// ML-powered &nbsp;·&nbsp; TF-IDF + Cosine Similarity &nbsp;·&nbsp; NLP Skill Extraction</div>"))

# ── STEP 1 ────────────────────────────────────────────────────
display(HTML("<span class='ra-label'>① Upload PDF Resume(s)</span>"))
display(HTML("<div class='step-note'>📄 1 resume → fit check &nbsp;·&nbsp; 📄📄 Multiple resumes → ranking mode</div>"))

file_upload = widgets.FileUpload(
    accept='.pdf', multiple=True,
    layout=widgets.Layout(width='100%', margin='8px 0')
)

load_btn = widgets.Button(
    description='Load Resumes',
    layout=widgets.Layout(width='180px', height='38px', margin='4px 0'),
    style={'button_color': '#1e1e35', 'font_weight': 'bold'}
)

display(file_upload)
display(load_btn)
display(load_status)

def on_load(b):
    with load_status:
        clear_output()
        resume_store.clear()

        # Support both old dict format and new tuple format
        raw = file_upload.value
        files = {}

        if isinstance(raw, dict):
            files = raw
        elif isinstance(raw, (list, tuple)):
            for item in raw:
                if hasattr(item, 'name'):
                    files[item.name] = {'content': item.content}
                elif isinstance(item, dict):
                    files[item.get('name', 'resume.pdf')] = item

        if not files:
            display(HTML("<div style='color:#ef4444;font-family:DM Mono,monospace;font-size:0.8rem;margin-top:6px'>⚠️ No files detected. Please upload a PDF first, then click Load.</div>"))
            return

        pills = ""
        for fname, fdata in files.items():
            content = fdata['content'] if isinstance(fdata, dict) else bytes(fdata)
            text = extract_text(io.BytesIO(bytes(content)))
            name = fname.replace('.pdf', '').replace('_', ' ').title()
            resume_store[name] = text
            pills += "<span style='display:inline-flex;align-items:center;gap:6px;background:#1a1a2e;border:1px solid #2a2a4a;border-radius:20px;padding:4px 12px;margin:3px;font-size:0.72rem;font-family:DM Mono,monospace;color:#a78bfa'>📄 " + fname + "</span>"

        count = len(resume_store)
        mode  = "Fit Check Mode" if count == 1 else "Ranking Mode (" + str(count) + " resumes)"
        display(HTML("<div style='margin:8px 0'>" + pills + "</div>"))
        display(HTML(
            "<div style='color:#00e5ff;font-family:DM Mono,monospace;font-size:0.78rem'>"
            + "✅ " + str(count) + " resume(s) loaded &nbsp;·&nbsp; "
            + "<span style='color:#a78bfa'>" + mode + "</span></div>"
        ))

load_btn.on_click(on_load)

# ── STEP 2 ────────────────────────────────────────────────────
display(HTML("<hr class='ra-divider'><span class='ra-label'>② Select Job Description</span>"))

jd_dropdown = widgets.Dropdown(
    options=["-- Select a job role --"] + list(JOB_DESCRIPTIONS.keys()),
    layout=widgets.Layout(width='100%', margin='4px 0')
)
display(jd_dropdown)
display(jd_status)

def on_jd_change(change):
    with jd_status:
        clear_output()
        val = jd_dropdown.value
        if val == "-- Select a job role --":
            return
        jd_skills = extract_skills(JOB_DESCRIPTIONS[val])
        tags = ''.join(["<span class='skill-tag'>" + s + "</span>" for s in jd_skills])
        display(HTML(
            "<div style='margin-top:8px;background:#071318;border:1px solid #00e5ff22;"
            "border-radius:10px;padding:12px 16px'>"
            + "<div style='color:#00e5ff;font-family:Syne,sans-serif;font-size:0.75rem;"
            "font-weight:700;margin-bottom:8px'>" + val + "</div>"
            + "<div style='font-family:DM Mono,monospace;font-size:0.7rem;color:#333;margin-bottom:8px'>Skills being evaluated:</div>"
            + "<div>" + (tags if tags else "<span style='color:#333;font-size:0.75rem'>none detected</span>") + "</div>"
            + "</div>"
        ))

jd_dropdown.observe(on_jd_change, names='value')

# ── STEP 3 ────────────────────────────────────────────────────
display(HTML("<hr class='ra-divider'><span class='ra-label'>③ Analyze</span>"))

analyze_btn = widgets.Button(
    description='⚡  Analyze Resumes',
    layout=widgets.Layout(width='200px', height='44px', margin='4px 0 16px 0'),
    style={'button_color': '#7c3aed', 'font_weight': 'bold'}
)
display(analyze_btn)
display(output_area)

def render_single(r):
    skill_tags = ''.join(
        ["<span class='skill-tag matched'>" + s + "</span>" for s in r['matched']]
        + ["<span class='skill-tag'>" + s + "</span>" for s in r['skills'] if s not in r['matched']]
    ) or "<span style='color:#2a2a3f;font-size:0.75rem;font-family:DM Mono,monospace'>No skills detected</span>"

    missing_tags = ''.join(["<span class='skill-tag missing'>" + s + "</span>" for s in r['missing'][:8]])
    skill_pct = round(len(r['matched']) / r['jd_skill_count'] * 100) if r['jd_skill_count'] > 0 else 0

    display(HTML(
        "<div class='ra-card'>"
        + "<div class='ra-card-top'>"
        + "<div style='font-size:2.2rem'>👤</div>"
        + "<div class='ra-info'>"
        + "<div class='ra-cname'>" + r['name']
        + "<span class='ra-badge'>" + str(r['years']) + " yrs exp</span>"
        + "<span class='ra-badge'>" + str(len(r['matched'])) + "/" + str(r['jd_skill_count']) + " skills</span>"
        + "</div>"
        + "<div style='margin-top:8px'>" + skill_tags + "</div>"
        + "</div>"
        + "<div class='ra-score-bar'>"
        + "<div class='ra-score-num'>" + str(r['score']) + "%</div>"
        + "<div class='bar-bg'><div class='bar-fill' style='width:" + str(r['score']) + "%'></div></div>"
        + "<div style='font-size:0.62rem;color:#2a2a3f;font-family:DM Mono,monospace;margin-top:4px'>match score</div>"
        + "</div>"
        + "</div>"
        + "<div class='fit-box' style='background:" + r['fit_bg'] + ";border:1px solid " + r['fit_border'] + "'>"
        + "<div class='fit-verdict' style='color:" + r['fit_color'] + "'>" + r['fit_label'] + "</div>"
        + ("<span style='color:#555'>Matched: </span><span style='color:#a78bfa'>" + ', '.join(r['matched']) + "</span><br>" if r['matched'] else "")
        + ("<span style='color:#555'>Missing: </span><span style='color:#ef4444'>" + ', '.join(r['missing'][:6]) + "</span><br>" if r['missing'] else "")
        + "<span style='color:#444'>Skill coverage: </span><span style='color:" + r['fit_color'] + "'>" + str(skill_pct) + "% of required skills found</span>"
        + "</div>"
        + (("<div style='margin-top:10px'><div style='font-family:DM Mono,monospace;font-size:0.7rem;color:#333;margin-bottom:4px'>Suggested skills to add:</div>" + missing_tags + "</div>") if missing_tags else "")
        + "</div>"
    ))

def render_multiple(results):
    rank_emojis = ['🥇', '🥈', '🥉']
    winner = results[0]
    display(HTML(
        "<div class='winner-banner'>🏆 Best Match: <span style='color:#00e5ff;font-weight:800'>"
        + winner['name'] + "</span>"
        + " &nbsp;·&nbsp; <span style='color:#a78bfa'>" + str(winner['score']) + "% match</span>"
        + " &nbsp;·&nbsp; " + str(len(winner['matched'])) + " skills matched"
        + " &nbsp;·&nbsp; " + str(winner['years']) + " yrs exp"
        + "</div>"
    ))
    for i, r in enumerate(results):
        emoji = rank_emojis[i] if i < 3 else ('#' + str(i + 1))
        skill_tags = ''.join(
            ["<span class='skill-tag matched'>" + s + "</span>" for s in r['matched']]
            + ["<span class='skill-tag'>" + s + "</span>" for s in r['skills'] if s not in r['matched']]
        ) or "<span style='color:#2a2a3f;font-size:0.75rem;font-family:DM Mono,monospace'>No skills detected</span>"
        display(HTML(
            "<div class='ra-card'>"
            + "<div class='ra-card-top'>"
            + "<div class='ra-rank' style='font-size:1.4rem;min-width:36px'>" + emoji + "</div>"
            + "<div class='ra-info'>"
            + "<div class='ra-cname'>" + r['name']
            + "<span class='ra-badge'>" + str(r['years']) + " yrs</span>"
            + "<span class='ra-badge'>" + str(len(r['matched'])) + "/" + str(r['jd_skill_count']) + " skills</span>"
            + "<span style='margin-left:8px;font-size:0.72rem;color:" + r['fit_color'] + ";font-family:DM Mono,monospace'>" + r['fit_label'] + "</span>"
            + "</div>"
            + "<div style='margin-top:8px'>" + skill_tags + "</div>"
            + "</div>"
            + "<div class='ra-score-bar'>"
            + "<div class='ra-score-num'>" + str(r['score']) + "%</div>"
            + "<div class='bar-bg'><div class='bar-fill' style='width:" + str(r['score']) + "%'></div></div>"
            + "</div>"
            + "</div>"
            + "</div>"
        ))

def on_analyze(b):
    with output_area:
        clear_output()
        if not resume_store:
            display(HTML("<div class='ra-empty'>⚠️ No resumes loaded. Upload PDFs and click <b>Load Resumes</b> first.</div>"))
            return
        if jd_dropdown.value == "-- Select a job role --":
            display(HTML("<div class='ra-empty'>⚠️ Please select a job description first.</div>"))
            return

        jd_text = JOB_DESCRIPTIONS[jd_dropdown.value]
        results = analyze_resumes(resume_store, jd_text)

        display(HTML(
            "<div style='font-family:DM Mono,monospace;font-size:0.76rem;color:#444;margin-bottom:12px'>"
            + "Analyzing: <span style='color:#a78bfa'>" + jd_dropdown.value + "</span>"
            + " &nbsp;·&nbsp; " + str(len(results)) + " resume(s)</div>"
        ))

        if len(results) == 1:
            render_single(results[0])
        else:
            render_multiple(results)

        display(HTML(
            "<div style='font-family:DM Mono,monospace;font-size:0.68rem;color:#222;margin-top:14px;text-align:center'>"
            + "Purple = matched skills &nbsp;·&nbsp; Cyan = other skills &nbsp;·&nbsp; Red = missing skills</div>"
        ))

analyze_btn.on_click(on_analyze)